In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [24]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [25]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [26]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    prompt = f'Generate an detailed outline for a blog on the topic {title}'
    outline = model.invoke(prompt).content

    state['outline'] = outline

    return state

In [27]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f"Write a detailed blog on the title - {title} using the following outline \n {outline}"
    content = model.invoke(prompt).content
    state['content'] = content

    return state

In [28]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)


graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)


workflow = graph.compile()

In [ ]:
initial_state = {'title': "rise of Ai in Pakistan"} 
final_state = workflow.invoke(initial_state)
print(final_state)